In [1]:
import pandas as pd

df = pd.read_csv('cpi_product_by_group.csv', skiprows=8, header=0)
print(df.head())
print(df.shape)

                         Geography        Canada     Unnamed: 2  Unnamed: 3  \
0  Products and product groups 3 4  January 2019  February 2019  March 2019   
1                              NaN      2002=100            NaN         NaN   
2                        All-items         133.6          134.5       135.4   
3                           Food 5         148.7          149.3       149.4   
4                        Shelter 6         143.0          143.4       143.8   

   Unnamed: 4 Unnamed: 5 Unnamed: 6 Unnamed: 7   Unnamed: 8      Unnamed: 9  \
0  April 2019   May 2019  June 2019  July 2019  August 2019  September 2019   
1         NaN        NaN        NaN        NaN          NaN             NaN   
2       136.0      136.6      136.3      137.0        136.8           136.2   
3       149.0      149.7      150.7      151.6        151.1           150.2   
4       144.1      144.2      144.0      144.3        144.6           144.8   

   ... Unnamed: 78 Unnamed: 79  Unnamed: 80     Un

In [2]:
# Clean the data
df.columns = ['product_group'] + list(df.iloc[0, 1:])
df = df.drop(index=[0, 1]).reset_index(drop=True)
df = df[['product_group'] + [col for col in df.columns if col != 'product_group']]

# Reshape from wide to long format
df_melted = df.melt(id_vars='product_group', var_name='month_year', value_name='cpi_value')

# Remove nulls and clean up
df_melted = df_melted.dropna()
df_melted['product_group'] = df_melted['product_group'].str.replace(r'\s+\d+$', '', regex=True).str.strip()
df_melted['cpi_value'] = pd.to_numeric(df_melted['cpi_value'], errors='coerce')
df_melted = df_melted.dropna()

print(df_melted.head(10))
print(df_melted.shape)

                                       product_group    month_year  cpi_value
0                                          All-items  January 2019      133.6
1                                               Food  January 2019      148.7
2                                            Shelter  January 2019      143.0
3    Household operations, furnishings and equipment  January 2019      123.3
4                              Clothing and footwear  January 2019       92.2
5                                     Transportation  January 2019      136.6
6                                           Gasoline  January 2019      149.6
7                           Health and personal care  January 2019      125.9
8                  Recreation, education and reading  January 2019      113.3
9  Alcoholic beverages, tobacco products and recr...  January 2019      170.6
(1305, 3)


In [3]:
df_melted.to_csv('cpi_clean.csv', index=False)
print("File saved successfully")

File saved successfully


In [5]:
!pip install mysql-connector-python

  Obtaining dependency information for mysql-connector-python from https://files.pythonhosted.org/packages/75/68/1f7469669ba1b7d70bec4076766f7672190021090d6eb7e9a0ca6097f501/mysql_connector_python-9.7.0-cp311-cp311-win_amd64.whl.metadata
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB 217.9 kB/s eta 0:01:21
   ---------------------------------------- 0.0/17.7 MB 245.8 kB/s eta 0:01:12
   ---------------------------------------- 0.2/17.7 MB 766.6 kB/s eta 0:00:23
   - -------------------------------------- 0.8/17.7 MB 3.2 MB/s eta 0:00:06
   --- ------------------------------------ 1.6/17.7 MB 5.8 MB/s eta 0:00:03
   ----- ---------------------------------- 2.6/17.7 MB 7.8 MB/s eta 0:00:02
   ---------- 

In [ ]:
import pandas as pd
import mysql.connector

conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='your_password_here',
    database='canada_cpi'
)

cursor = conn.cursor()

df = pd.read_csv('cpi_clean.csv')

for _, row in df.iterrows():
    cursor.execute(
        "INSERT INTO cpi_data VALUES (%s, %s, %s)",
        (row['product_group'], row['month_year'], float(row['cpi_value']))
    )

conn.commit()
print("Data loaded successfully")